Cài thư viện + import

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import xgboost as xgb
import joblib

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from google.colab import files

import warnings
warnings.filterwarnings("ignore")

RANDOM_STATE = 42

Upload

In [ ]:
uploaded = files.upload()

Saving sample_submission.csv to sample_submission (2).csv
Saving train.csv to train (2).csv
Saving test.csv to test (2).csv


Đọc dữ liệu

In [ ]:
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")
sample_submission = pd.read_csv("sample_submission.csv")

print("Train:", train_df.shape)
print("Test:", test_df.shape)
print("Sample submission:", sample_submission.shape)

display(train_df.head())

Train: (1460, 81)
Test: (1459, 80)
Sample submission: (1459, 2)


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


Tách biến mục tiêu

In [ ]:
TARGET = "SalePrice"

X = train_df.drop(columns=[TARGET])
y = train_df[TARGET]

X_test = test_df.copy()

print("X:", X.shape)
print("y:", y.shape)
print("X_test:", X_test.shape)

X: (1460, 80)
y: (1460,)
X_test: (1459, 80)


Preprocessing

In [ ]:
all_data = pd.concat(
    [X, X_test],
    axis=0,
    ignore_index=True
)

print("Before preprocessing:", all_data.shape)

Before preprocessing: (2919, 80)


In [ ]:
categorical_cols = all_data.select_dtypes(
    include=["object"]
).columns

numerical_cols = all_data.select_dtypes(
    include=["int64", "float64"]
).columns

# Categorical
all_data[categorical_cols] = all_data[categorical_cols].fillna("None")

# Numerical
for col in numerical_cols:
    all_data[col] = all_data[col].fillna(
        all_data[col].median()
    )

print(
    "Missing values:",
    all_data.isnull().sum().sum()
)

Missing values: 0


In [ ]:
all_data = pd.get_dummies(
    all_data,
    columns=categorical_cols,
    drop_first=False
)

print("After encoding:", all_data.shape)

After encoding: (2919, 311)


In [ ]:
X_processed = all_data.iloc[:len(X)].copy()
X_test_processed = all_data.iloc[len(X):].copy()

X_processed = X_processed.astype(float)
X_test_processed = X_test_processed.astype(float)
y = y.astype(float)

print("Processed train:", X_processed.shape)
print("Processed test:", X_test_processed.shape)

Processed train: (1460, 311)
Processed test: (1459, 311)


In [ ]:
feature_names = X_processed.columns.tolist()

joblib.dump(
    feature_names,
    "feature_names.pkl"
)

print("Đã lưu feature_names.pkl")

Đã lưu feature_names.pkl


In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X_processed,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE
)

print("Training:", X_train.shape)
print("Validation:", X_val.shape)

Training: (1168, 311)
Validation: (292, 311)


In [ ]:
def evaluate_model(model, X_val, y_val):

    pred = model.predict(X_val)

    rmse = np.sqrt(
        mean_squared_error(y_val, pred)
    )

    mae = mean_absolute_error(
        y_val,
        pred
    )

    r2 = r2_score(
        y_val,
        pred
    )

    return rmse, mae, r2

MODEL XGBoost

In [ ]:
xgb_model = xgb.XGBRegressor(
    objective="reg:squarederror",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

xgb_model.fit(
    X_train,
    y_train
)

print("XGBoost training completed!")

XGBoost training completed!


In [ ]:
xgb_rmse, xgb_mae, xgb_r2 = evaluate_model(
    xgb_model,
    X_val,
    y_val
)

print("===== XGBoost =====")
print("RMSE:", xgb_rmse)
print("MAE :", xgb_mae)
print("R²  :", xgb_r2)

===== XGBoost =====
RMSE: 28087.415276084965
MAE : 17701.74119755993
R²  : 0.8971487162890092


Tuning XGBoost

In [ ]:
xgb_params = {
    "n_estimators": [1500, 2000, 2500, 3000],
    "max_depth": [2, 3, 4],
    "learning_rate": [0.01, 0.02, 0.03],
    "min_child_weight": [1, 2, 3, 5],
    "subsample": [0.7, 0.8, 0.9],
    "colsample_bytree": [0.7, 0.8, 0.9],
    "reg_alpha": [0, 0.01, 0.05, 0.1],
    "reg_lambda": [0.5, 1.0, 1.5],
    "gamma": [0, 0.01, 0.05]
}

In [ ]:
xgb_random = RandomizedSearchCV(
    estimator=xgb.XGBRegressor(
        objective="reg:squarederror",
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),

    param_distributions=xgb_params,

    n_iter=30,

    scoring="neg_root_mean_squared_error",

    cv=5,

    random_state=RANDOM_STATE,

    verbose=1,

    n_jobs=-1
)

xgb_random.fit(
    X_train,
    y_train
)

Fitting 5 folds for each of 30 candidates, totalling 150 fits


RandomizedSearchCV(cv=5,
                   estimator=XGBRegressor(base_score=None, booster=None,
                                          callbacks=None,
                                          colsample_bylevel=None,
                                          colsample_bynode=None,
                                          colsample_bytree=None, device=None,
                                          early_stopping_rounds=None,
                                          enable_categorical=True,
                                          eval_metric=None, feature_types=None,
                                          feature_weights=None, gamma=None,
                                          grow_policy=None,
                                          importance_type=None,
                                          interaction_constraints...
                   n_iter=30, n_jobs=-1,
                   param_distributions={'colsample_bytree': [0.7, 0.8, 0.9],
                                        'gamma': [0, 0.01, 0.05],
                                        'learning_rate': [0.01, 0.02, 0.03],
                                        'max_depth': [2, 3, 4],
                                        'min_child_weight': [1, 2, 3, 5],
                                        'n_estimators': [1500, 2000, 2500,
                                                         3000],
                                        'reg_alpha': [0, 0.01, 0.05, 0.1],
                                        'reg_lambda': [0.5, 1.0, 1.5],
                                        'subsample': [0.7, 0.8, 0.9]},
                   random_state=42, scoring='neg_root_mean_squared_error',
                   verbose=1)

In [ ]:
best_xgb = xgb_random.best_estimator_

print("Best XGBoost parameters:")
print(xgb_random.best_params_)

Best XGBoost parameters:
{'subsample': 0.9, 'reg_lambda': 0.5, 'reg_alpha': 0, 'n_estimators': 3000, 'min_child_weight': 1, 'max_depth': 2, 'learning_rate': 0.03, 'gamma': 0, 'colsample_bytree': 0.8}


In [ ]:
xgb_rmse, xgb_mae, xgb_r2 = evaluate_model(
    best_xgb,
    X_val,
    y_val
)

print("\n===== BEST XGBOOST =====")
print("RMSE:", xgb_rmse)
print("MAE :", xgb_mae)
print("R²  :", xgb_r2)


===== BEST XGBOOST =====
RMSE: 24740.876149473777
MAE : 15568.783323523116
R²  : 0.920197533180525


MODEL Decision Tree

In [ ]:
dt_params = {
    "max_depth": [8, 10, 12, 14, 16],
    "min_samples_split": [2, 3, 5, 8, 10],
    "min_samples_leaf": [1, 2, 3, 4],
    "max_features": [0.6, 0.7, 0.8, 0.9, 1.0]
}

In [ ]:
dt_random = RandomizedSearchCV(
    estimator=DecisionTreeRegressor(
        random_state=RANDOM_STATE
    ),

    param_distributions=dt_params,

    n_iter=30,

    scoring="neg_root_mean_squared_error",

    cv=5,

    random_state=RANDOM_STATE,

    verbose=1,

    n_jobs=-1
)

dt_random.fit(
    X_train,
    y_train
)

Fitting 5 folds for each of 30 candidates, totalling 150 fits


RandomizedSearchCV(cv=5, estimator=DecisionTreeRegressor(random_state=42),
                   n_iter=30, n_jobs=-1,
                   param_distributions={'max_depth': [8, 10, 12, 14, 16],
                                        'max_features': [0.6, 0.7, 0.8, 0.9,
                                                         1.0],
                                        'min_samples_leaf': [1, 2, 3, 4],
                                        'min_samples_split': [2, 3, 5, 8, 10]},
                   random_state=42, scoring='neg_root_mean_squared_error',
                   verbose=1)

In [ ]:
best_dt = dt_random.best_estimator_

print("Best Decision Tree parameters:")
print(dt_random.best_params_)

Best Decision Tree parameters:
{'min_samples_split': 10, 'min_samples_leaf': 3, 'max_features': 0.7, 'max_depth': 14}


In [ ]:
dt_rmse, dt_mae, dt_r2 = evaluate_model(
    best_dt,
    X_val,
    y_val
)

print("\n===== BEST DECISION TREE =====")
print("RMSE:", dt_rmse)
print("MAE :", dt_mae)
print("R²  :", dt_r2)


===== BEST DECISION TREE =====
RMSE: 39026.34337684141
MAE : 24805.910143548474
R²  : 0.8014353710354555


MODEL Random Forest

In [ ]:
rf_params = {
    "n_estimators": [800, 1000, 1200, 1500],
    "max_depth": [None, 15, 20, 25, 30],
    "min_samples_split": [2, 3, 5, 8],
    "min_samples_leaf": [1, 2, 3],
    "max_features": [0.6, 0.7, 0.8, 0.9],
    "bootstrap": [True]
}

In [ ]:
 rf_random = RandomizedSearchCV(
    estimator=RandomForestRegressor(
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),

    param_distributions=rf_params,

    n_iter=30,

    scoring="neg_root_mean_squared_error",

    cv=5,

    random_state=RANDOM_STATE,

    verbose=1,

    n_jobs=-1
)

rf_random.fit(
    X_train,
    y_train
)

Fitting 5 folds for each of 30 candidates, totalling 150 fits


RandomizedSearchCV(cv=5,
                   estimator=RandomForestRegressor(n_jobs=-1, random_state=42),
                   n_iter=30, n_jobs=-1,
                   param_distributions={'bootstrap': [True],
                                        'max_depth': [None, 15, 20, 25, 30],
                                        'max_features': [0.6, 0.7, 0.8, 0.9],
                                        'min_samples_leaf': [1, 2, 3],
                                        'min_samples_split': [2, 3, 5, 8],
                                        'n_estimators': [800, 1000, 1200,
                                                         1500]},
                   random_state=42, scoring='neg_root_mean_squared_error',
                   verbose=1)

In [ ]:
best_rf = rf_random.best_estimator_

print("Best Random Forest parameters:")
print(rf_random.best_params_)

Best Random Forest parameters:
{'n_estimators': 800, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 0.6, 'max_depth': 25, 'bootstrap': True}


In [ ]:
rf_rmse, rf_mae, rf_r2 = evaluate_model(
    best_rf,
    X_val,
    y_val
)

print("\n===== BEST RANDOM FOREST =====")
print("RMSE:", rf_rmse)
print("MAE :", rf_mae)
print("R²  :", rf_r2)


===== BEST RANDOM FOREST =====
RMSE: 28859.475309734367
MAE : 17056.854997893563
R²  : 0.8914167021511072


Train lại 3 model trên toàn bộ train.csv

In [ ]:
print("Training final XGBoost...")
best_xgb.fit(
    X_processed,
    y
)

print("Training final Decision Tree...")
best_dt.fit(
    X_processed,
    y
)

print("Training final Random Forest...")
best_rf.fit(
    X_processed,
    y
)

print("\nAll final models trained!")

Training final XGBoost...
Training final Decision Tree...
Training final Random Forest...

All final models trained!


In [ ]:
joblib.dump(
    best_xgb,
    "xgboost_model.pkl"
)

joblib.dump(
    best_dt,
    "decision_tree_model.pkl"
)

joblib.dump(
    best_rf,
    "random_forest_model.pkl"
)

print("Đã lưu 3 model:")
print("1. xgboost_model.pkl")
print("2. decision_tree_model.pkl")
print("3. random_forest_model.pkl")

Đã lưu 3 model:
1. xgboost_model.pkl
2. decision_tree_model.pkl
3. random_forest_model.pkl


Dự đoán test.csv

In [ ]:
pred_xgb = best_xgb.predict(
    X_test_processed
)

pred_dt = best_dt.predict(
    X_test_processed
)

pred_rf = best_rf.predict(
    X_test_processed
)

print("Đã dự đoán test.csv")

Đã dự đoán test.csv


Tạo kết quả submission

In [ ]:
submission_xgb = sample_submission.copy()

submission_xgb["SalePrice"] = pred_xgb

submission_xgb.to_csv(
    "submission_xgb.csv",
    index=False
)

In [ ]:
submission_dt = sample_submission.copy()

submission_dt["SalePrice"] = pred_dt

submission_dt.to_csv(
    "submission_decision_tree.csv",
    index=False
)

In [ ]:
submission_rf = sample_submission.copy()

submission_rf["SalePrice"] = pred_rf

submission_rf.to_csv(
    "submission_random_forest.csv",
    index=False
)

Kiểm tra kết quả

In [ ]:
print("===== XGBOOST =====")
display(submission_xgb.head())

print("===== DECISION TREE =====")
display(submission_dt.head())

print("===== RANDOM FOREST =====")
display(submission_rf.head())

===== XGBOOST =====


,Id,SalePrice
0,1461,123380.750000
1,1462,166928.000000
2,1463,190488.703125
3,1464,194438.796875
4,1465,182919.046875


===== DECISION TREE =====


,Id,SalePrice
0,1461,106071.428571
1,1462,145181.250000
2,1463,205750.000000
3,1464,205750.000000
4,1465,213722.285714


===== RANDOM FOREST =====


,Id,SalePrice
0,1461,126693.624583
1,1462,154436.705000
2,1463,178948.724107
3,1464,183731.067091
4,1465,198055.607500


In [ ]:
model_info = {
    "target": TARGET,
    "feature_names": feature_names,
    "random_state": RANDOM_STATE
}

joblib.dump(
    model_info,
    "model_info.pkl"
)

print("Đã lưu model_info.pkl")

Đã lưu model_info.pkl


In [40]:
import shutil
import zipfile
from google.colab import files
import os

files_to_save = [
    "xgboost_model.pkl",
    "decision_tree_model.pkl",
    "random_forest_model.pkl",
    "feature_names.pkl",
    "model_info.pkl",
    "submission_xgb.csv",
    "submission_decision_tree.csv",
    "submission_random_forest.csv"
]

# Create the zip containing only the listed files
zip_name = "house_price_models_and_results.zip"

with zipfile.ZipFile(zip_name, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for f in files_to_save:
        if os.path.exists(f):
            zf.write(f)
            print(f"Added: {f}")
        else:
            print(f"Warning: {f} not found, skipped")

# Download the zip
files.download(zip_name)

Added: xgboost_model.pkl
Added: decision_tree_model.pkl
Added: random_forest_model.pkl
Added: feature_names.pkl
Added: model_info.pkl
Added: submission_xgb.csv
Added: submission_decision_tree.csv
Added: submission_random_forest.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>